# Nixtla

https://github.com/Nixtla/nixtla

In [1]:
# !pip install statsforecast neuralforecast mlforecast

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, MSTL
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, LSTM

In [3]:
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (15, 5)

In [4]:
def evaluate(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'MSE': mean_squared_error(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred),
    }

In [5]:
fname = 'train.csv'
df = pd.read_csv(fname, parse_dates=['date'])

# Filter store and item
store, item = 1, 1
df = df[(df['store'] == store) & (df['item'] == item)][['date', 'sales']]
df = df.rename(columns={'date': 'ds', 'sales': 'y'})

# Add unique_id column (Nixtla required format)
df['unique_id'] = 'sales'

print(f"Data shape: {df.shape}")
print(f"Date range: {df['ds'].min()} to {df['ds'].max()}")
print(df.head())

Data shape: (1826, 3)
Date range: 2013-01-01 00:00:00 to 2017-12-31 00:00:00
          ds   y unique_id
0 2013-01-01  13     sales
1 2013-01-02  11     sales
2 2013-01-03  14     sales
3 2013-01-04  13     sales
4 2013-01-05  10     sales


In [6]:
test_size = 365
train_df = df.iloc[:-test_size].copy()
test_df = df.iloc[-test_size:].copy()
y_true = test_df['y'].values

print(f"\nTrain size: {len(train_df)}, Test size: {len(test_df)}")


Train size: 1461, Test size: 365


## STATSFORECAST (Statistical Models)

In [7]:
models_sf = [
    AutoARIMA(season_length=7),      # weekly seasonality
    AutoETS(season_length=7),        # exponential smoothing
    MSTL(season_length=[7, 365]),    # multiple seasonality (weekly + yearly)
]

sf = StatsForecast(
    models=models_sf,
    freq='D',              # daily data
    n_jobs=-1,
)

# Fit and predict
sf.fit(train_df)
sf_pred = sf.predict(h=test_size).reset_index()
sf_pred = sf_pred.rename(columns={'ds': 'ds_pred'})

# Extract predictions
arima_pred = sf_pred[sf_pred['unique_id'] == 'sales']['AutoARIMA'].values
ets_pred = sf_pred[sf_pred['unique_id'] == 'sales']['AutoETS'].values
mstl_pred = sf_pred[sf_pred['unique_id'] == 'sales']['MSTL'].values

## MLFORECAST (Machine Learning Models) 

In [8]:
# Define lag features
lags = [1, 2, 3, 4, 5, 7, 14, 28, 365]

# Define transformations on lags (rolling statistics)
lag_transforms = {
    1: [('rolling_mean', 7), ('rolling_std', 7)],
    7: [('rolling_mean', 28)],
    14: [('rolling_mean', 28)],
}

# Date features
date_features = ['dayofweek', 'month', 'dayofyear', 'quarter', 'weekofyear']

# Target transforms (optional - try with/without differencing)
target_transforms = [Differences([1])]

# Models
models_ml = {
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'CatBoost': CatBoostRegressor(verbose=0, random_seed=42),
}

fcst = MLForecast(
    models=models_ml,
    freq='D',
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=date_features,
    target_transforms=target_transforms,
    num_threads=4,
)

# Train
fcst.fit(train_df)
ml_pred = fcst.predict(test_size)

# Extract predictions
rf_pred = ml_pred[ml_pred['unique_id'] == 'sales']['RandomForest'].values
cat_pred = ml_pred[ml_pred['unique_id'] == 'sales']['CatBoost'].values

AssertionError: 

## NEURALFORECAST (Deep Learning Models)

In [ ]:
horizon = test_size
input_size = 2 * horizon  # context length

models_nf = [
    NHITS(
        h=horizon,
        input_size=input_size,
        max_steps=100,
        n_freq_downsample=[2, 1, 1],
    ),
    LSTM(
        h=horizon,
        input_size=input_size,
        max_steps=100,
        encoder_hidden_size=64,
        decoder_hidden_size=64,
    ),
]

nf = NeuralForecast(models=models_nf, freq='D')
nf.fit(train_df, val_size=horizon)  # use test period for validation

# Cross-validation prediction (predict test period)
Y_hat_df = nf.predict()

# Filter for test period only
y_dates = pd.to_datetime(Y_hat_df['ds'].unique())
test_dates = pd.to_datetime(test_df['ds'].unique())
mask = Y_hat_df['ds'].isin(test_dates)
nf_filtered = Y_hat_df[mask]

# Extract predictions for sales series
nhits_pred = nf_filtered[nf_filtered['unique_id'] == 'sales']['NHITS'].values
lstm_pred = nf_filtered[nf_filtered['unique_id'] == 'sales']['LSTM'].values

In [ ]:
results = []
results.append(evaluate(y_true, arima_pred, 'AutoARIMA'))
results.append(evaluate(y_true, ets_pred, 'AutoETS'))
results.append(evaluate(y_true, mstl_pred, 'MSTL'))
results.append(evaluate(y_true, rf_pred, 'RandomForest'))
results.append(evaluate(y_true, cat_pred, 'CatBoost'))
results.append(evaluate(y_true, nhits_pred, 'NHITS'))
results.append(evaluate(y_true, lstm_pred, 'LSTM'))

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('MSE')
print(comparison_df.to_string(float_format='%.2f'))

In [ ]:
# Create a dictionary of predictions for plotting
all_predictions = {
    'AutoARIMA': arima_pred,
    'AutoETS': ets_pred,
    'MSTL': mstl_pred,
    'RandomForest': rf_pred,
    'CatBoost': cat_pred,
    'NHITS': nhits_pred,
    'LSTM': lstm_pred,
}

# Select best and worst models for visualization
best_model = comparison_df.iloc[0]['Model']
worst_model = comparison_df.iloc[-1]['Model']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Best model
axes[0].plot(train_df['ds'], train_df['y'], label='Train', color='blue')
axes[0].plot(test_df['ds'], y_true, label='Actual', color='green')
axes[0].plot(test_df['ds'], all_predictions[best_model], label=f'{best_model}', linestyle='--', color='red')
axes[0].set_title(f'Best Model: {best_model}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Worst model
axes[1].plot(train_df['ds'], train_df['y'], label='Train', color='blue')
axes[1].plot(test_df['ds'], y_true, label='Actual', color='green')
axes[1].plot(test_df['ds'], all_predictions[worst_model], label=f'{worst_model}', linestyle='--', color='red')
axes[1].set_title(f'Worst Model: {worst_model}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Loading data...
Data shape: (1826, 3)
Date range: 2013-01-01 00:00:00 to 2017-12-31 00:00:00
          ds   y unique_id
0 2013-01-01  13     sales
1 2013-01-02  11     sales
2 2013-01-03  14     sales
3 2013-01-04  13     sales
4 2013-01-05  10     sales

Train size: 1461, Test size: 365

Training StatsForecast models...

Training MLForecast models...


AssertionError: 